# BarterBench — Kaggle Community Benchmark

**BarterBench** places LLM agents in competitive N-agent marketplace scenarios with designed resource scarcity. Agents must trade items to achieve conflicting goals — not all can succeed.

This benchmark tests **social cognition**: specifically, whether a model can suppress cooperative disclosure norms when the competitive context demands it.

### Key finding to beat
In 505 runs across 18 models, the Information Security Score (ISS) averages 54% — models reveal their private goals roughly half the time, handing competitors a decisive advantage. The random baseline achieves ~61% goal completion; most LLMs fail to beat it.

GitHub: https://github.com/JamesEBall/BarterBench

In [ ]:
# Download BarterBench source files directly (no git required)
import urllib.request, os

BASE = 'https://raw.githubusercontent.com/JamesEBall/BarterBench/main/'
FILES = ['agent.py', 'engine.py', 'scoring.py', 'bradley_terry.py',
         'elo.py', 'model_registry.py', 'scenario_gen.py', 'solvability.py', 'taxonomy.py']
SCENARIOS = ['gold_rush', 'water_crisis', 'spice_wars', 'grand_bazaar']

os.makedirs('scenarios', exist_ok=True)
for f in FILES:
    urllib.request.urlretrieve(BASE + f, f)
for s in SCENARIOS:
    urllib.request.urlretrieve(BASE + f'scenarios/{s}.json', f'scenarios/{s}.json')

# Install Python dependencies
!pip install -q openai anthropic python-dotenv

In [ ]:
import kaggle_benchmarks as kbench
import json, random, sys
from pathlib import Path

from agent import (
    MarketAgent, RandomAgent,
    _build_marketplace_context, _parse_json_response,
    JSON_SCHEMA_INSTRUCTION,
)
from engine import MarketEngine
from scoring import compute_metrics

SCENARIOS_DIR = Path('scenarios')

def load_scenario(name):
    return json.loads((SCENARIOS_DIR / f'{name}.json').read_text())

In [ ]:
class KbenchAgent(MarketAgent):
    """MarketAgent wrapper that delegates LLM calls to a kbench llm object."""

    def __init__(self, kbench_llm, agent_idx, **kwargs):
        super().__init__(model_name='kbench', agent_idx=agent_idx,
                         backend='cli', **kwargs)
        self._kbench_llm = kbench_llm

    def take_turn(self, inventory, target, order_book, recent_trades,
                  round_num, max_rounds, auctions=None):
        context = _build_marketplace_context(
            self.agent_idx, inventory, target, order_book, recent_trades,
            round_num, max_rounds,
            strategy_prompt=self.strategy_prompt,
            auctions=auctions,
        )
        prompt = f"{context}\n\n{JSON_SCHEMA_INSTRUCTION}\n\nIt's your turn. Choose your action."
        try:
            raw = self._kbench_llm.prompt(prompt)
            raw = raw.strip()
            if '```' in raw:
                raw = raw.split('```')[1].lstrip('json').strip()
            action = _parse_json_response(raw)
        except Exception:
            action = self._fallback_pass()
        self._record_round_history(action, round_num)
        return action


def run_match(kbench_llm, scenario_name: str, seed: int) -> dict:
    """Run one match. Agent 0 is the LLM under test; all others are random.
    Single LLM agent keeps quota usage minimal while preserving competitive dynamics.
    Returns {gc: float, iss: float} for the LLM agent.
    """
    scenario = load_scenario(scenario_name)
    num_agents = len(scenario['agents'])

    agents = [KbenchAgent(kbench_llm, agent_idx=0)]  # only agent 0 is LLM
    for i in range(1, num_agents):
        agents.append(RandomAgent(agent_idx=i, seed=seed + i))

    engine = MarketEngine(scenario, agents, run_id=seed, live_updates=False)
    result = engine.run()
    result['scenario_data'] = scenario
    metrics = compute_metrics(result)

    gc  = metrics.get('model_goal_completion', {}).get('kbench', 0)
    iss = metrics.get('information_security_score', {}).get('kbench', 0)
    return {'gc': gc, 'iss': iss}

In [ ]:
# ── Task definitions ──────────────────────────────────────────────────────────

@kbench.task(name='gold_rush')
def gold_rush(llm, seed: int):
    """6-agent gold rush marketplace. LLM controls 3 agents vs 3 random opponents."""
    r = run_match(llm, 'gold_rush', seed=seed)
    kbench.assertions.assert_greater_than(
        r['gc'], 0.0,
        expectation=f"LLM agents should complete at least some goal (got {r['gc']:.1%} GC, ISS={r['iss']:.1%})"
    )


@kbench.task(name='water_crisis')
def water_crisis(llm, seed: int):
    """8-agent water crisis. LLM controls 4 agents vs 4 random opponents."""
    r = run_match(llm, 'water_crisis', seed=seed)
    kbench.assertions.assert_greater_than(
        r['gc'], 0.0,
        expectation=f"LLM agents should complete at least some goal (got {r['gc']:.1%} GC, ISS={r['iss']:.1%})"
    )


@kbench.task(name='spice_wars')
def spice_wars(llm, seed: int):
    """10-agent spice wars. LLM controls 5 agents vs 5 random opponents."""
    r = run_match(llm, 'spice_wars', seed=seed)
    kbench.assertions.assert_greater_than(
        r['gc'], 0.0,
        expectation=f"LLM agents should complete at least some goal (got {r['gc']:.1%} GC, ISS={r['iss']:.1%})"
    )

In [ ]:
# 3 seeds × 3 scenarios = 9 tasks per model (~27 LLM calls per model, ~81 total)
SEEDS = list(range(3))

for seed in SEEDS:
    gold_rush.run(llm=kbench.llm, seed=seed)

for seed in SEEDS:
    water_crisis.run(llm=kbench.llm, seed=seed)

for seed in SEEDS:
    spice_wars.run(llm=kbench.llm, seed=seed)